# **Обучение CatBoost модели и её применение**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 26-03-2026  

**Цель:** Обучение CatBoost модели, примение модели на тестовой выборке для получения результатов в kaggle соревновании ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview), а также сбор всех необходимых для дашборда метрик

#### Необходимые библиотеки и настройка графиков:

In [1]:
%pip install catboost -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import gzip
import zipfile
from collections import Counter
from datetime import datetime

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, classification_report

In [3]:
%config InlineBackend.figure_format = 'retina'

sns.set(style='darkgrid', palette='deep')

plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'

### 0. Загрука набора данных c kaggle
Скачиваем все файлы с соревнования ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) для дальнейшего использования.

In [4]:
os.environ['KAGGLE_API_TOKEN'] = "KGAT_4263738b1187d00be6cd3d9c9162ed8f" # ВАШ_KAGGLE_API_TOKEN

print("✅ Kaggle API ключ установлен!")

✅ Kaggle API ключ установлен!


In [5]:
%pip install kaggle -q
!kaggle competitions download -c avazu-ctr-prediction -p ../

Note: you may need to restart the kernel to use updated packages.
avazu-ctr-prediction.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
with zipfile.ZipFile('../avazu-ctr-prediction.zip', 'r') as zip_ref:
    zip_ref.extractall('../avazu-ctr-prediction')

### 1. Обучение CatBoost модели

Обучаться будем батчами по $10$ млн. строк, так как датасет слишком большой, чтобы целиком храниться в памяти при обучении.

При обучении будем использовать Out-Of-Time Validation, это лучше отражает качество моделей на времязависимых данных. Нам повезло и набор данных хранится уже осторированным. Как мы узнали ранее, в датасете $40 428 967$ строк. Оставим первые $30$ млн. из них на обучение (то есть получится 3 батча), остальные олтложим для валидации. Получим, что примерно $74$% от тренировочного набора данных уйдет на обучающую выборку и $26$% на валидацонную.

Все признаки в датасете категориальные, кроме `hour`. Но этот признак нельзя передавать как численный: так мы неявно зададим порядок по времени. Я предлагаю вытащить из колонки 2 категориальных признака: номер часа в сутках и номер дня недели. Если оставить 2 признака числовыми, то мы опять сталкнемся с проблеммой неявного попрядка на данных.

In [7]:
def data_tranformer(df: pd.DataFrame, drop_target: bool = False):
    dt = pd.to_datetime(df['hour'], format='%y%m%d%H')
    df['day_of_week'] = dt.dt.dayofweek
    df['hour_of_day'] = dt.dt.hour

    columns = ['id', 'hour']
    if drop_target:
        columns.append('click')

    return df.drop(columns=columns, axis=1).astype(str)

#### Перейдем к обучению

In [11]:
os.makedirs(f'models', exist_ok=True)
model_path = 'models/catboost_ctr_model.cbm'

target = 'click'
categorical_features = [
    'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip',
    'device_model', 'device_type', 'device_conn_type',
    'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21',
    'day_of_week', 'hour_of_day' # новые признаки, которые мы создадим сами
]

model_params = {
    'iterations': 150,
    'learning_rate': 0.08,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 67,
    'task_type': 'CPU', # GPU не поддреживает батчи(
    'thread_count': -1, # Идем на взлет
    'verbose': 50
}

chunk_size = 10_000_000
train_file = '../avazu-ctr-prediction/train.gz'
chunk_iterator = pd.read_csv(train_file, compression='gzip', chunksize=chunk_size)

val_size = 0
print("🚀 Начало обучения (3 чанка по 10 млн)...")
for chunk_num, chunk in enumerate(chunk_iterator, 1):
    if chunk_num <= 3:
        print(f"\n📦 Чанк №{chunk_num} - обучение (Строки: {chunk.index.min():_} - {chunk.index.max():_})".replace('_', ' '))

        y_chunk = chunk[target]
        X_chunk = data_tranformer(chunk, True)

        current_model = CatBoostClassifier(**model_params)

        if chunk_num == 1:
            current_model.fit(X_chunk, y_chunk, cat_features=categorical_features)
        else:
            current_model.fit(X_chunk, y_chunk, cat_features=categorical_features, init_model=model_path)

        current_model.save_model(model_path)
        del current_model, chunk, X_chunk, y_chunk
        gc.collect()
    else:
        print(f"\n{'='*55}")
        print("🧪 Чтение оставшихся данных для OOT-Валидации")

        remaining_chunks = [chunk] + list(chunk_iterator)
        val_df = pd.concat(remaining_chunks, ignore_index=True)

        del chunk, remaining_chunks, chunk_iterator
        gc.collect()

        val_size = len(val_df)
        print(f"Валидационная выборка собрана: {len(val_df):_} строк".replace('_', ' '))

        val_y_true = val_df[target]
        val_X = data_tranformer(val_df, True)

        print("⏳ Генерация предсказаний для валидационной выборки...")
        val_model = CatBoostClassifier()
        val_model.load_model(model_path)

        val_y_pred_proba = val_model.predict_proba(val_X)[:, 1]
        val_y_pred_class = val_model.predict(val_X)

        del val_model, val_X
        gc.collect()

        break

print("\n🎉 Обучение завершено!!!")

🚀 Начало обучения (3 чанка по 10 млн)...

📦 Чанк №1 - обучение (Строки: 0 - 9 999 999)
0:	total: 4.16s	remaining: 10m 20s
50:	total: 1m 30s	remaining: 2m 55s
100:	total: 2m 54s	remaining: 1m 24s
149:	total: 4m 15s	remaining: 0us

📦 Чанк №2 - обучение (Строки: 10 000 000 - 19 999 999)
0:	total: 3.52s	remaining: 8m 44s
50:	total: 1m 27s	remaining: 2m 49s
100:	total: 2m 47s	remaining: 1m 21s
149:	total: 4m 5s	remaining: 0us

📦 Чанк №3 - обучение (Строки: 20 000 000 - 29 999 999)
0:	total: 3.71s	remaining: 9m 13s
50:	total: 1m 25s	remaining: 2m 45s
100:	total: 2m 43s	remaining: 1m 19s
149:	total: 3m 59s	remaining: 0us

🧪 Чтение оставшихся данных для OOT-Валидации
Валидационная выборка собрана: 10 428 967 строк
⏳ Генерация предсказаний для валидационной выборки...

🎉 Обучение завершено!!!


Обучение и валидация прошли успешно, теперь посмотрим на получившиеся на валидационной выборке метрики:

In [12]:
roc_auc = roc_auc_score(val_y_true, val_y_pred_proba)
logloss = log_loss(val_y_true, val_y_pred_proba)
acc = accuracy_score(val_y_true, val_y_pred_class)

print(f"📊 Итоговые метрики (OOT выборка на {f'{val_size:_}'.replace('_', ' ')} строк):")

metrics_df = pd.DataFrame({
    'Метрика': ['ROC-AUC Score', 'Log Loss', 'Accuracy'],
    'Значение': [roc_auc, logloss, acc]
})
metrics_df['Значение'] = metrics_df['Значение'].round(4)

metrics_df

📊 Итоговые метрики (OOT выборка на 10 428 967 строк):


,Метрика,Значение
0,ROC-AUC Score,0.7074
1,Log Loss,0.4196
2,Accuracy,0.8295


### 2. Применение модели на тестовой выборке и отправка решения на kaggle

Считаем тестовую выборку:

In [13]:
warnings.filterwarnings('ignore')

test_file = "../avazu-ctr-prediction/test.gz"

print("⏳ Читаем test.gz...")
test_df = pd.read_csv(test_file, compression='gzip', dtype={'id': str})

print(f"✅ Итого загружено: {f"{len(test_df):_}".replace('_', ' ')} строк")

ram_usage = test_df.memory_usage(deep=True).sum() / 1024**3
print(f"📊 Объем памяти DataFrame: {ram_usage:.2f} GB")

⏳ Читаем test.gz...
✅ Итого загружено: 4 577 464 строк
📊 Объем памяти DataFrame: 2.92 GB


А также модель:

In [14]:
print("⏳ Достаем обученную модель...")
model = CatBoostClassifier()
model.load_model(model_path)

print(f"Количество признаков в модели: {len(model.feature_names_)}")

⏳ Достаем обученную модель...
Количество признаков в модели: 23


Применим модель к тестовому набору данных:

In [ ]:
ids = test_df['id']
X_test = data_tranformer(test_df)

del test_df
gc.collect()

print("\n⏳ Генерация предсказаний...")
y_pred_proba = model.predict_proba(X_test)[:, 1]

del X_test
gc.collect()

NameError: name 'dt' is not defined

Выведем минимальную статистику по полученным предсказаниям:

In [ ]:
stats_data = [
    ('Средняя вероятность', y_pred_proba.mean()),
    ('Медианная вероятность', np.median(y_pred_proba)),
    ('Стандартное отклонение', y_pred_proba.std()),
    ('Минимум', y_pred_proba.min()),
    ('Максимум', y_pred_proba.max())
]
stats_df = pd.DataFrame(stats_data, columns=['Статистика', 'Значение'])
stats_df['Значение'] = stats_df['Значение'].round(4)

print("📊 Статистики предсказаний на тестовой выборке:")
display(stats_df)


quantiles = []
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    quantiles.append((f'{int(q*100)}%', np.quantile(y_pred_proba, q)))
quantiles_df = pd.DataFrame(quantiles, columns=['Квантиль', 'Значение'])

print("📊 Квантили на тестовой выборке:")
display(quantiles_df)


📊 Статистики предсказаний на тестовой выборке:


,Статистика,Значение
0,Средняя вероятность,0.2591
1,Медианная вероятность,0.2655
2,Стандартное отклонение,0.0768
3,Минимум,0.0956
4,Максимум,0.5531


📊 Квантили на тестовой выборке:


,Квантиль,Значение
0,10%,0.163077
1,25%,0.203883
2,50%,0.265505
3,75%,0.295502
4,90%,0.325899


*Наконец, сохраненим результат и сделаем посылку submission'а на kaggle:*

In [ ]:
submission_file = 'data/submission.csv'
os.makedirs(f'data', exist_ok=True)

In [ ]:
submission = pd.DataFrame({'id': ids, 'click': y_pred_proba})
submission.to_csv(submission_file, index=False)
print(f"✅ Результаты успешно сохранены в: {submission_file}")

✅ Результаты успешно сохранены в: data/submission.csv


In [ ]:
!kaggle competitions submit -c avazu-ctr-prediction -f data/submission.csv  -m "CatBoost submission"

100%|████████████████████████████████████████| 174M/174M [00:55<00:00, 3.30MB/s]
Successfully submitted to Click-Through Rate Prediction

### 3. Сбор всех необходимых метрик и данных для дашборда

Решение отправленно, теперь надо подготовить и сохранить все данные для посторения 3-й вкладки дашборда в Yandex DataLens.

##### Базовые метрики на валидационной выборке:

In [ ]:
metrics_df = pd.DataFrame({
    'metric': ['ROC-AUC', 'Log Loss', 'Accuracy'],
    'value': [roc_auc, logloss, accuracy]
})
metrics_df.to_csv('data/model_metrics.csv', index=False)
print("✅ Базовые метрики сохранены: data/model_metrics.csv")

##### Данные для Lift:

In [ ]:
# TODO: Возможно унести в DataLens расчеты
print("⏳ Расчет Lift-метрик...")

global_ctr = val_df[target].mean()
total_traffic = len(val_df)

lift_data = []

for col in categorical_features:
    grouped = val_df.groupby(col)[target].agg(['count', 'mean']).reset_index()
    grouped.rename(columns={col: 'Значение колонки', 'count': 'Трафик', 'mean': 'Локальный CTR'}, inplace=True)
    grouped['Название колонки'] = col
    grouped['Процент трафика'] = (grouped['Трафик'] / total_traffic) * 100

    grouped['Прибавка к CTR (%)'] = (grouped['Локальный CTR'] - global_ctr) * 100

    lift_data.append(grouped[['Название колонки', 'Значение колонки', 'Прибавка к CTR (%)', 'Процент трафика']])

lift_df = pd.concat(lift_data, ignore_index=True)

lift_df['Прибавка к CTR (%)'] = lift_df['Прибавка к CTR (%)']
lift_df['Процент трафика'] = lift_df['Процент трафика']


lift_df.to_csv('data/lift_data.csv', index=False)
print("✅ Данные для графика Lift успешно сохранены в data/lift_data.csv")

display(lift_df.head())

##### Feature Importance:

In [ ]:
feature_importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

importance_df.to_csv('data/feature_importance.csv', index=False)
print("✅ Важность признаков сохранена в: data/feature_importance.csv")

##### Предсказанный CTR для каждого `id`:

In [ ]:
print(f"⏳ Потоковая загрузка для '{train_file}'...")
chunk_size = 10_000_000

chunk_iterator = pd.read_csv(train_file, compression='gzip', chunksize=chunk_size, dtype={'id': str})

output_file = 'data/train_ctr_predictions.csv'
for chunk_num, df in enumerate(chunk_iterator, 1):
    print(f"\n📦 Чанк {chunk_num} (Строки: {df.index.min():_} - {df.index.max():_})".replace('_', ' '))

    ids = df['id']
    X = data_tranformer(df)

    print("⏳ Генерация предсказаний...")
    y_pred_proba = model.predict_proba(X)[:, 1]

    del X, df
    gc.collect()

    chunk_res = pd.DataFrame({'id': ids, 'click': y_pred_proba})

    print("💾 Запись на диск...")
    if chunk_num == 1:
        chunk_res.to_csv(output_file, index=False, mode='w')
    else:
        chunk_res.to_csv(output_file, index=False, mode='a', header=False)

    del chunk_res, ids
    gc.collect()

print(f"\n✅ Вероятность клика для каждого показа из train.gz успешно сохранена в: {output_file}")

: 

##### Данные для диаграммы надежности модели:

In [ ]:
# TODO: Возможно тоже унести в DataLens
from sklearn.calibration import calibration_curve

print("⏳ Расчет данных для диаграммы надежности (Calibration Curve)...")

# TODO: узнать про то, как будет влиять разбиение на train и val на дашборд
prob_true, prob_pred = calibration_curve(val_y_true, val_y_pred_proba, n_bins=10, strategy='uniform')

calibration_df = pd.DataFrame({
    'Предсказанная вероятность': prob_pred,
    'Фактический CTR': prob_true
})

calibration_df['Идеальная калибровка'] = calibration_df['Предсказанная вероятность']

print("📊 Координаты для графика калибровки:")
display(calibration_df)

calibration_df.to_csv(data/calibration_curve.csv, index=False)
print(f"\n✅ Данные успешно сохранены в: {data/calibration_curve.csv}")